# Mini-Projet Data Mining 2026/2027 — Étape 1 & 2 : Prétraitement, Encodage & Prévention des Fuites
**Auteurs (Codes Étudiants) :** RSD04 (Team Leader), CS12, CS22, CS29, IASD05, IASD25  
**Parcours :** Classification Supervisée  
**Contrainte Clé :** Absence absolue de fuite de données (*Data Leakage Prevention*) — toutes les transformations sont encapsulées dans un `ColumnTransformer` scikit-learn ajusté uniquement sur le train set.

---
## 1. Architecture du Pipeline de Prétraitement
1. **Découpage stratifié Train/Test :** 80% Train, 20% Test, `stratify=y`, `random_state=42`.
2. **Sous-pipeline Variables Numériques :**
   - Imputation des valeurs manquantes par la médiane (`SimpleImputer(strategy='median')`).
   - Normalisation Z-Score (`StandardScaler()`).
3. **Sous-pipeline Variable Ordinale (`Class`) :**
   - Ordre explicite : `Economy` < `Economy Plus` < `Business`.
   - Encodage ordinal (`OrdinalEncoder`) suivi d'une standardisation.
4. **Sous-pipeline Variables Nominales (`Gender`, `Customer Type`, `Type of Travel`) :**
   - Encodage disjonctif One-Hot (`OneHotEncoder(drop='first', handle_unknown='ignore')`).
5. **Sauvegarde des jeux transformés et de l'objet préprocesseur scikit-learn (`joblib`).**


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import joblib

sys.path.append('..')
sys.path.append('.')
from src.preprocessing import load_raw_data, clean_data, build_preprocessor, prepare_data_splits, get_feature_names

df_raw = load_raw_data()
print(f"Données brutes chargées : {df_raw.shape}")


Données brutes chargées : (129880, 24)


## 2. Découpage Stratifié et Ajustement du Préprocesseur


In [2]:
splits = prepare_data_splits(df_raw, test_size=0.20, random_state=42)

X_train_raw = splits['X_train_raw']
X_test_raw = splits['X_test_raw']
X_train_trans = splits['X_train_trans']
X_test_trans = splits['X_test_trans']
y_train = splits['y_train']
y_test = splits['y_test']
preprocessor = splits['preprocessor']

print(f"Train set brut : {X_train_raw.shape} -> transformé : {X_train_trans.shape}")
print(f"Test set brut  : {X_test_raw.shape} -> transformé : {X_test_trans.shape}")
print(f"Proportion de classe 1 (Satisfait) dans Train : {y_train.mean():.4f}")
print(f"Proportion de classe 1 (Satisfait) dans Test  : {y_test.mean():.4f}")


Train set brut : (103904, 22) -> transformé : (103904, 22)
Test set brut  : (25976, 22) -> transformé : (25976, 22)
Proportion de classe 1 (Satisfait) dans Train : 0.4345
Proportion de classe 1 (Satisfait) dans Test  : 0.4345


## 3. Inspection des Caractéristiques Transformées


In [3]:
feature_names = get_feature_names(preprocessor)
print(f"Nombre de caractéristiques finales : {len(feature_names)}")
print("Liste des caractéristiques :", feature_names)

df_train_sample = pd.DataFrame(X_train_trans[:5], columns=feature_names)
df_train_sample


Nombre de caractéristiques finales : 22
Liste des caractéristiques : ['Age', 'Flight Distance', 'Departure Delay', 'Arrival Delay', 'Departure and Arrival Time Convenience', 'Ease of Online Booking', 'Check-in Service', 'Online Boarding', 'Gate Location', 'On-board Service', 'Seat Comfort', 'Leg Room Service', 'Cleanliness', 'Food and Drink', 'In-flight Service', 'In-flight Wifi Service', 'In-flight Entertainment', 'Baggage Handling', 'Class', 'Gender_Male', 'Customer Type_Returning', 'Type of Travel_Personal']


,Age,Flight Distance,Departure Delay,Arrival Delay,Departure and Arrival Time Convenience,Ease of Online Booking,Check-in Service,Online Boarding,Gate Location,On-board Service,...,Cleanliness,Food and Drink,In-flight Service,In-flight Wifi Service,In-flight Entertainment,Baggage Handling,Class,Gender_Male,Customer Type_Returning,Type of Travel_Personal
0,1.095776,0.658793,-0.385030,-0.389948,0.616134,0.884235,-0.241098,-0.927536,0.798213,0.479113,...,-1.739422,0.600106,0.303593,0.953912,0.481376,0.312327,1.008293,1.0,1.0,0.0
1,2.019955,0.180855,-0.306583,-0.389948,1.270669,0.170805,-0.241098,0.553591,-0.766882,-1.852802,...,0.543491,0.600106,-2.246218,0.201260,-1.768395,-2.228876,-1.070040,0.0,1.0,1.0
2,0.567674,1.285022,-0.385030,-0.389948,-0.038401,-1.256056,1.337718,0.553591,0.015665,1.256417,...,1.304462,0.600106,1.153530,0.201260,1.231300,1.159395,1.008293,0.0,1.0,0.0
3,1.557866,-0.279047,0.660925,0.336191,-1.347472,0.170805,-1.030506,-0.186973,0.015665,0.479113,...,1.304462,1.352776,-1.396281,0.201260,1.231300,-1.381809,-1.070040,1.0,1.0,1.0
4,1.161789,-0.629735,-0.071244,-0.389948,0.616134,0.884235,-0.241098,0.553591,0.798213,0.479113,...,-0.217480,0.600106,0.303593,0.953912,0.481376,0.312327,1.008293,1.0,1.0,0.0


## 4. Vérification de l'absence de valeurs manquantes résiduelles


In [4]:
assert not np.isnan(X_train_trans).any(), "Erreur : Des NaN subsistent dans X_train_trans !"
assert not np.isnan(X_test_trans).any(), "Erreur : Des NaN subsistent dans X_test_trans !"
print("-> Vérification réussie : 0 valeur manquante dans les matrices prétraitées !")


-> Vérification réussie : 0 valeur manquante dans les matrices prétraitées !


## 5. Export des données préparées et du préprocesseur


In [5]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

train_df = pd.DataFrame(X_train_trans, columns=feature_names)
train_df['Satisfaction'] = y_train.values
train_df.to_csv("../data/processed/train_processed.csv.gz", compression='gzip', index=False)

test_df = pd.DataFrame(X_test_trans, columns=feature_names)
test_df['Satisfaction'] = y_test.values
test_df.to_csv("../data/processed/test_processed.csv.gz", compression='gzip', index=False)

joblib.dump(preprocessor, "../models/preprocessor.joblib")
print("Préprocesseur et jeux de données sauvegardés avec succès dans data/processed/ et models/ !")


Préprocesseur et jeux de données sauvegardés avec succès dans data/processed/ et models/ !
